In [0]:
   %pip install plotly

In [0]:
from pyspark.sql import functions as F, Window

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")
T = "workspace.heatcheck."

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = [F.col(c) for c in (lat1, lon1, lat2, lon2)]
    return 6371 * 2 * F.asin(F.sqrt(
        F.pow(F.sin(F.radians(lat2 - lat1) / 2), 2) +
        F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) *
        F.pow(F.sin(F.radians(lon2 - lon1) / 2), 2)))

def rank_scale(df, col, out):
    """Rank each subzone against all others (0.1 = lowest, 1 = highest)."""
    return df.withColumn(out, 0.1 + 0.9 * F.percent_rank().over(Window.orderBy(col)))

wbgt = spark.table(T + "silver_wbgt")

# Heat: average daily peak WBGT per station over the last year
daily = (wbgt.filter(F.col("reading_time") >= F.date_sub(F.current_date(), 365))
    .groupBy("station_id", F.to_date("reading_time").alias("day"))
    .agg(F.max("wbgt").alias("peak_wbgt")))
hazard = daily.groupBy("station_id").agg(
    F.avg("peak_wbgt").alias("avg_peak_wbgt"),
    F.sum(F.when(F.col("peak_wbgt") >= 33, 1).otherwise(0)).alias("high_heat_days"),
    F.count("*").alias("days_observed"))
stations = (wbgt.filter(F.col("latitude").isNotNull())
    .groupBy("station_id")
    .agg(F.first("station_name").alias("station_name"),
         F.avg("latitude").alias("st_lat"), F.avg("longitude").alias("st_lon"))
    .join(hazard, "station_id"))

# Seniors + subzone centre points
subzones = spark.table(T + "silver_subzone_centroids")
seniors = (spark.table(T + "silver_seniors_by_subzone")
    .withColumn("subzone_upper", F.upper(F.trim("subzone")))
    .join(subzones, "subzone_upper")
    .filter(F.col("seniors_65plus") >= 100))

# Nearest WBGT station to each subzone
w = Window.partitionBy("subzone_upper").orderBy("dist_station_km")
nearest_station = (seniors.select("subzone_upper", "sz_lat", "sz_lon").crossJoin(stations)
    .withColumn("dist_station_km", haversine_km("sz_lat", "sz_lon", "st_lat", "st_lon"))
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1")
    .select("subzone_upper", "station_name", "avg_peak_wbgt", "high_heat_days", "dist_station_km"))

# Distance to the nearest eldercare point
elder = spark.table(T + "silver_eldercare_points")
nearest_elder = (seniors.select("subzone_upper", "sz_lat", "sz_lon").crossJoin(elder)
    .withColumn("d", haversine_km("sz_lat", "sz_lon", "lat", "lon"))
    .groupBy("subzone_upper").agg(F.min("d").alias("dist_eldercare_km")))

risk = (seniors.join(nearest_station, "subzone_upper").join(nearest_elder, "subzone_upper")
    .withColumn("share_small_flats", F.col("seniors_small_flats") / F.col("seniors_65plus")))

# Heat x Vulnerable seniors x Distance from help
risk = rank_scale(risk, "seniors_small_flats", "E")     # how many seniors live in 1-2 room flats
risk = rank_scale(risk, "dist_eldercare_km", "V")       # farther from eldercare = less support
hs = risk.agg(F.min("avg_peak_wbgt").alias("mn"), F.max("avg_peak_wbgt").alias("mx")).first()
risk = risk.withColumn("H", 0.7 + 0.3 * (F.col("avg_peak_wbgt") - hs.mn) / ((hs.mx - hs.mn) or 1))
risk = (risk
    .withColumn("risk_raw", F.col("H") * F.col("E") * (0.6 + 0.4 * F.col("V")))
    .withColumn("risk_index", F.round(100 * F.percent_rank().over(Window.orderBy("risk_raw"))).cast("int"))
    .withColumn("risk_band",
        F.when(F.col("risk_index") >= 90, "Check first")
         .when(F.col("risk_index") >= 70, "Check next")
         .otherwise("Monitor"))
    .withColumn("risk_rank", F.row_number().over(Window.orderBy(F.desc("risk_raw")))))

(risk.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(T + "gold_subzone_risk"))
print("Subzones scored:", risk.count())
print(risk.groupBy("risk_band").count().orderBy("risk_band").collect())
display(risk.select("risk_rank", "risk_band", "planning_area", "subzone", "risk_index",
                    "seniors_65plus", "seniors_small_flats",
                    F.round("share_small_flats", 2).alias("share_small"),
                    F.round("avg_peak_wbgt", 1).alias("avg_peak_wbgt"),
                    F.round("dist_eldercare_km", 2).alias("km_to_eldercare"))
        .orderBy("risk_rank").limit(15))

In [0]:
import json, re
import plotly.express as px

T = "workspace.heatcheck."

def kml_attrs(desc):
    return dict(re.findall(r"<th>(.*?)</th>\s*<td>(.*?)</td>", desc or ""))

geo = json.load(open("/Volumes/workspace/heatcheck/raw/reference/subzones.geojson"))
for feat in geo["features"]:
    if "SUBZONE_N" not in feat["properties"]:
        feat["properties"] = kml_attrs(feat["properties"].get("Description"))

pdf = spark.table(T + "gold_subzone_risk").select(
    "subzone_upper", "subzone", "planning_area", "risk_index", "risk_band",
    "seniors_65plus", "seniors_small_flats").toPandas()

fig = px.choropleth_map(
    pdf, geojson=geo, locations="subzone_upper", featureidkey="properties.SUBZONE_N",
    color="risk_index", color_continuous_scale="YlOrRd", range_color=(0, 100),
    hover_data=["risk_band", "planning_area", "subzone", "seniors_65plus", "seniors_small_flats"],
    map_style="carto-positron", center={"lat": 1.3521, "lon": 103.8198}, zoom=10.3,
    opacity=0.8, template="plotly_white",
    title="HeatCheck v0: where seniors are most at risk from heat (by subzone)")
fig.update_layout(height=650, margin=dict(l=0, r=0, t=50, b=0),
                  coloraxis_colorbar=dict(title="Risk index"))
fig.show()